# Predicitive Analysis- Model Training (REGRESSION)

In this notebook, we will be training a Random Forest Regressor model for machine failure prediction. We begin by dropping the NaN values for our RUL column, then train a Random Forest Regressor model on the dataset. After training, we evaluate the model's perfomance on the validation and final datasets and export the model for deployment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PATH = '/content/drive/MyDrive/predictive-analysis-data/regression/'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor

### Loading the dataset

In [ ]:
# loading the dataframes
train_df = pd.read_csv(f"{PATH}train_regression.csv")
val_df = pd.read_csv(f"{PATH}val_regression.csv")
test_df = pd.read_csv(f"{PATH}test_regression.csv")

need to drop NaN values. only training on datawith 24 rul. this works in production also because we will only be running regression on the model failing in next 24 hours.

We are dropping the NaN values in the RUL_hours column and only train the model on data with RUL information within 24 hours. This will work in our production as we will only be running the regression on the machines failure in next 24 hours.

In [ ]:
train_df = train_df.dropna(subset=["RUL_hours"])
val_df = val_df.dropna(subset=["RUL_hours"])
test_df = test_df.dropna(subset=["RUL_hours"])

In [ ]:
# excluding metadata & failure target variable from dataset
exclude_cols = ['machineID', 'datetime', 'RUL_hours']

feature_cols = [col for col in train_df.columns if col not in exclude_cols]

X_train = train_df[feature_cols]
y_train = train_df['RUL_hours']

X_val = val_df[feature_cols]
y_val = val_df['RUL_hours']

X_test = test_df[feature_cols]
y_test = test_df['RUL_hours']

In [ ]:
# storing metadata for later analysis
train_meta = train_df[['machineID', 'datetime', 'RUL_hours']].copy()
val_meta = val_df[['machineID', 'datetime', 'RUL_hours']].copy()
test_meta = test_df[['machineID', 'datetime', 'RUL_hours']].copy()

### Peek into our data

In [ ]:
print(f"\nDataset Summary:")
print(f"Train samples: {len(X_train):,}")
print(f"Val samples: {len(X_val):,}")
print(f"Test samples: {len(X_test):,}")
print(f"Total features: {X_train.shape[1]}")

print(f"\nRUL (Remaining Useful Life) Distribution:")
print(f"\nTrain Set:")
print(f"  Mean:   {y_train.mean():.2f} hours")
print(f"  Median: {y_train.median():.2f} hours")
print(f"  Std:    {y_train.std():.2f} hours")
print(f"  Min:    {y_train.min():.2f} hours")
print(f"  Max:    {y_train.max():.2f} hours")

print(f"\nValidation Set:")
print(f"  Mean:   {y_val.mean():.2f} hours")
print(f"  Median: {y_val.median():.2f} hours")
print(f"  Std:    {y_val.std():.2f} hours")
print(f"  Min:    {y_val.min():.2f} hours")
print(f"  Max:    {y_val.max():.2f} hours")

print(f"\nTest Set:")
print(f"  Mean:   {y_test.mean():.2f} hours")
print(f"  Median: {y_test.median():.2f} hours")
print(f"  Std:    {y_test.std():.2f} hours")
print(f"  Min:    {y_test.min():.2f} hours")
print(f"  Max:    {y_test.max():.2f} hours")



Dataset Summary:
Train samples: 15,106
Val samples: 1,432
Test samples: 1,364
Total features: 59

RUL (Remaining Useful Life) Distribution:

Train Set:
  Mean:   11.98 hours
  Median: 12.00 hours
  Std:    7.19 hours
  Min:    0.00 hours
  Max:    24.00 hours

Validation Set:
  Mean:   11.96 hours
  Median: 12.00 hours
  Std:    7.22 hours
  Min:    0.00 hours
  Max:    24.00 hours

Test Set:
  Mean:   11.91 hours
  Median: 12.00 hours
  Std:    7.24 hours
  Min:    0.00 hours
  Max:    24.00 hours


In [ ]:
X_train.head()

,volt,rotate,pressure,vibration,age,error1,error2,error3,error4,error5,...,total_maintenances_to_date,total_failures_to_date,model2,model3,model4,age_squared,volt_deviation_from_machine_avg,rotate_deviation_from_machine_avg,pressure_deviation_from_machine_avg,vibration_deviation_from_machine_avg
72,-0.373016,0.034020,-0.283305,1.459761,1.144551,-0.034347,-0.033619,-0.031135,-0.028581,48.483120,...,-1.629882,-1.24155,-0.45257,1.36277,-0.685994,1.269857,-0.376665,0.039131,-0.266132,1.422669
73,-1.815080,-0.254706,1.635046,2.689368,1.144551,-0.034347,-0.033619,-0.031135,-0.028581,-0.020626,...,-1.629882,-1.24155,-0.45257,1.36277,-0.685994,1.269857,-1.818821,-0.249630,1.652928,2.652585
74,1.315276,0.626064,0.105924,2.323607,1.144551,-0.034347,-0.033619,-0.031135,-0.028581,-0.020626,...,-1.629882,-1.24155,-0.45257,1.36277,-0.685994,1.269857,1.311736,0.631246,0.123241,2.286731
75,-0.861835,-0.163241,1.113126,2.474843,1.144551,-0.034347,-0.033619,-0.031135,-0.028581,-0.020626,...,-1.629882,-1.24155,-0.45257,1.36277,-0.685994,1.269857,-0.865516,-0.158154,1.130815,2.438006
76,-1.138887,-0.123867,-0.572252,3.166037,1.144551,-0.034347,-0.033619,-0.031135,-0.028581,-0.020626,...,-1.629882,-1.24155,-0.45257,1.36277,-0.685994,1.269857,-1.142585,-0.118775,-0.555185,3.129373


In [ ]:
y_train.head()

,RUL_hours
72,24.0
73,23.0
74,22.0
75,21.0
76,20.0


### Training Random Forest Regressor Model



In [ ]:
# training the model
base_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

base_model.fit(X_train, y_train)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   17.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   40.2s finished


RandomForestRegressor(n_jobs=-1, random_state=42, verbose=1)

#### Evaluating the model on validation set (change for regression)

After training, we want to see how well the model performs on new unseen data. For now, we evaluate the model's performance on our validation dataset by caclulating these metrics: r2_score, mean_squared_error and mean_absolute_error. These performance metrics help us evaluate how well the model is making predictions. Unlike simple accuracy, which isn't applicable to regression tasks, these metrics provide a more meaningful assessment by focusing on the magnitude of errors and the quality of the model's fit.

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [ ]:
# get predictions on val dataset
y_val_prediction = base_model.predict(X_val)

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.0s finished


In [ ]:
# R-squared score
val_r2 = r2_score(y_val, y_val_prediction)

# Mean Absolute Error (less sensitive to outliers)
val_mae = mean_absolute_error(y_val, y_val_prediction)

# Root Mean Squared Error (error in original units)
# Use squared=False for RMSE
val_rmse = mean_squared_error(y_val, y_val_prediction)

print(f"\nR-squared (R2): {val_r2:.4f}")
print(f"Mean Absolute Error (MAE): {val_mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {val_rmse:.4f}")


R-squared (R2): 0.9953
Mean Absolute Error (MAE): 0.0816
Root Mean Squared Error (RMSE): 0.2447


From the performance metrics, we can see that our model performs exceptionally well with high accuracy. <br>The R² score of 0.9953 indicates that the model explains 99.53% of the variance in the target variable, showing a strong fit to the data. Meanwhile, the **Mean Absolute Error (MAE) of 0.0816** and **Root Mean Squared Error (RMSE) of 0.2447** suggest that the model's predictions are highly accurate, with minimal error. These results reflect the model's excellent performance in making precise predictions on unseen data.

### Feature Importance Analysis

To understand which features have the highest influence on our model's predicition (and on a machine's failure), we are performing a feature importance analysis on the trained Random Forest model. This maps and sorts the most important/influential features for the model's decision-making process.

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': base_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))


10 Most Important Features:
                     feature  importance
      hours_since_last_error    0.520135
hours_since_last_maintenance    0.406812
        error_count_last_24h    0.025782
        total_errors_to_date    0.016207
                 has_failure    0.004980
               rotate_lag_3h    0.002481
      rotate_rolling_24h_std    0.001798
             has_maintenance    0.001658
                 volt_lag_3h    0.001642
  vibration_rolling_24h_mean    0.001438


From the results, we can see that the model's predictions are primarily driven by the hours since the last error (52.01%) and hours since the last maintenance (40.68%). These two features together account for nearly 93% of the model's predictive power, indicating that the model places significant weight on recent operational history and maintenance intervals when predicting failure. Other features, such as error count in the last 24 hours and total errors to date, also contribute, but to a much lesser extent, reflecting that the model relies heavily on recent events to assess failure risk.

### Final Evaluation on test set

In [ ]:
# get predictions on test dataset
y_test_prediction = base_model.predict(X_test)

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.0s finished


In [ ]:
# calculate all metrics
test_r2 = r2_score(y_test, y_test_prediction)
test_mse = mean_squared_error(y_test, y_test_prediction)
test_mae = mean_absolute_error(y_test, y_test_prediction)

print(f"\nFinal test set perfomance metrics:")
print(f"R2 score: {test_r2:.4f}")
print(f"MSE: {test_mse:.4f}")
print(f"MAE: {test_mae:.4f}")


Final test set perfomance metrics:
R2 score: 0.9834
MSE: 0.8711
MAE: 0.1544


The final perfomance metrics on the test set confirms the Random Forest Regressor model's strong performance and reliability in estimating remaining useful life.<br>
The R² score of 0.9834 indicates that the model explains 98.34% of the variance in the data, reflecting excellent predictive power. The Mean Squared Error (MSE) of 0.8711 and Mean Absolute Error (MAE) of 0.1544 further confirm that the model's predictions are accurate, with minimal error. These results highlight the model's reliability and precision in estimating future failures with high confidence.

### Exporting trained model

In [ ]:
import os
import pickle

In [ ]:
# save the trained model
with open(f"{PATH}regression_model.pkl", 'wb') as f:
    pickle.dump(base_model, f)
print("Saved regression_model.pkl")

Saved regression_model.pkl
